# AdS4 Phase-Lock Critical Threshold Extraction (v10)

This notebook sharpens the identifiability sweep from v9.

We:
- sweep structured GEO corruption from 0.00 to 0.30 in steps of 0.02
- measure mean / max reconstruction error
- estimate a critical corruption threshold c*
- visualize low / near-threshold / high corruption reconstructions

Goal:
- turn the phase-lock breakdown into a measurable number
- prepare the way for probe-independence and completeness tests


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# Ground-truth AdS4-style toy background
r = torch.linspace(1.05, 3.5, 240, device=device).view(-1, 1)

def f_true(r):
    return r**2 + 1 - 1/r

def g_true(r):
    return 1/(f_true(r) + 1e-6)

def ee_true(r):
    return torch.log(1.0 + 0.6 * f_true(r))

def wl_true(r):
    return torch.sqrt(f_true(r) + 1.8 * g_true(r) + 1e-6)

def geo_true(r):
    return torch.sqrt(f_true(r) + 0.7 * g_true(r) + 1e-6)

ee_obs = ee_true(r).detach()
wl_obs = wl_true(r).detach()
geo_obs = geo_true(r).detach()


In [ ]:
def structured_geo_target(r, corruption_strength=0.0):
    base = geo_true(r)
    if corruption_strength == 0.0:
        return base.detach()
    radial_bias = 1.0 + corruption_strength * (0.30 * (r - r.min()) / (r.max() - r.min()))
    oscillation = 1.0 + corruption_strength * 0.15 * torch.sin(2.2 * r)
    return (base * radial_bias * oscillation).detach()


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = MLP()
        self.g = MLP()

    def forward(self, r):
        f = F.softplus(self.f(r))
        g = F.softplus(self.g(r))
        return f, g


In [ ]:
def train(corruption_strength=0.0, epochs=1200, consistency_weight=0.15):
    m = Model().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    geo_target = structured_geo_target(r, corruption_strength)

    loss_hist = []
    consistency_hist = []

    for _ in range(epochs):
        opt.zero_grad()
        f, g = m(r)

        ee_pred = torch.log(1.0 + 0.6 * f)
        wl_pred = torch.sqrt(f + 1.8 * g + 1e-6)
        geo_pred = torch.sqrt(f + 0.7 * g + 1e-6)

        loss_ee = ((ee_pred - ee_obs)**2).mean()
        loss_wl = ((wl_pred - wl_obs)**2).mean()
        loss_geo = ((geo_pred - geo_target)**2).mean()
        loss_consistency = ((f * g - 1.0)**2).mean()

        loss = loss_ee + loss_wl + loss_geo + consistency_weight * loss_consistency
        loss.backward()
        opt.step()

        loss_hist.append(loss.item())
        consistency_hist.append(loss_consistency.item())

    with torch.no_grad():
        f_pred, g_pred = m(r)
        metric_err = ((f_pred - f_true(r)).abs() + (g_pred - g_true(r)).abs()) / 2.0
        abs_err_mean = metric_err.mean().item()
        abs_err_max = metric_err.max().item()
        consistency_final = consistency_hist[-1]

    return {
        'loss_hist': loss_hist,
        'consistency_hist': consistency_hist,
        'f_pred': f_pred,
        'g_pred': g_pred,
        'abs_err_mean': abs_err_mean,
        'abs_err_max': abs_err_max,
        'consistency_final': consistency_final,
    }


In [ ]:
# Refined sweep
strengths = [round(x, 2) for x in np.arange(0.00, 0.301, 0.02)]
results = []

for s in strengths:
    out = train(corruption_strength=float(s), epochs=1200)
    results.append({
        'strength': float(s),
        'abs_err_mean': out['abs_err_mean'],
        'abs_err_max': out['abs_err_max'],
        'consistency_final': out['consistency_final'],
        'loss_hist': out['loss_hist'],
        'f_pred': out['f_pred'],
        'g_pred': out['g_pred'],
    })
    print(f"corruption={s:.2f} | mean err={out['abs_err_mean']:.6f} | max err={out['abs_err_max']:.6f} | consistency={out['consistency_final']:.6f}")


In [ ]:
# Critical threshold extraction
mean_tolerance = 0.10
max_tolerance = 0.10

c_star_mean = None
c_star_max = None

for row in results:
    if c_star_mean is None and row['abs_err_mean'] > mean_tolerance:
        c_star_mean = row['strength']
    if c_star_max is None and row['abs_err_max'] > max_tolerance:
        c_star_max = row['strength']

print('mean tolerance:', mean_tolerance)
print('max tolerance:', max_tolerance)
print('c*_mean:', c_star_mean)
print('c*_max:', c_star_max)


In [ ]:
# Plot critical threshold extraction
strength_vals = [row['strength'] for row in results]
mean_err_vals = [row['abs_err_mean'] for row in results]
max_err_vals = [row['abs_err_max'] for row in results]
consistency_vals = [row['consistency_final'] for row in results]

plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
plt.plot(strength_vals, mean_err_vals, marker='o', label='mean abs error')
plt.axhline(mean_tolerance, linestyle='--', label='mean tolerance')
if c_star_mean is not None:
    plt.axvline(c_star_mean, linestyle=':', label=f'c*_mean={c_star_mean:.2f}')
plt.xlabel('corruption strength')
plt.ylabel('mean abs error')
plt.title('mean threshold')
plt.legend()

plt.subplot(2, 2, 2)
plt.plot(strength_vals, max_err_vals, marker='o', color='tab:red', label='max abs error')
plt.axhline(max_tolerance, linestyle='--', label='max tolerance')
if c_star_max is not None:
    plt.axvline(c_star_max, linestyle=':', label=f'c*_max={c_star_max:.2f}')
plt.xlabel('corruption strength')
plt.ylabel('max abs error')
plt.title('max threshold')
plt.legend()

plt.subplot(2, 2, 3)
plt.plot(strength_vals, consistency_vals, marker='o', color='tab:green', label='consistency drift')
plt.xlabel('corruption strength')
plt.ylabel('final consistency loss')
plt.title('geometry class stability')
plt.legend()

plt.subplot(2, 2, 4)
idx_low = 0
idx_mid = len(results)//2
idx_high = len(results)-1
plt.plot(results[idx_low]['loss_hist'], label=f"low={results[idx_low]['strength']:.2f}")
plt.plot(results[idx_mid]['loss_hist'], label=f"mid={results[idx_mid]['strength']:.2f}")
plt.plot(results[idx_high]['loss_hist'], label=f"high={results[idx_high]['strength']:.2f}")
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('training under corruption')
plt.legend()

plt.suptitle('AdS4 Critical Threshold Extraction (v10)')
plt.tight_layout()
plt.savefig('ads4_phase_lock_v10_threshold.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v10_threshold.png')


In [ ]:
# Reconstruction comparison near low / c* / high corruption
def nearest_index(target):
    diffs = [abs(row['strength'] - target) for row in results]
    return int(np.argmin(diffs))

pick_low = nearest_index(0.00)
pick_mean = nearest_index(c_star_mean if c_star_mean is not None else 0.16)
pick_high = nearest_index(0.30)

plt.figure(figsize=(12, 4))

for i, idx in enumerate([pick_low, pick_mean, pick_high], start=1):
    row = results[idx]
    plt.subplot(1, 3, i)
    plt.plot(r.cpu(), f_true(r).cpu(), label='f true')
    plt.plot(r.cpu(), row['f_pred'].cpu(), '--', label=f"f @{row['strength']:.2f}")
    plt.plot(r.cpu(), g_true(r).cpu(), label='g true')
    plt.plot(r.cpu(), row['g_pred'].cpu(), ':', label=f"g @{row['strength']:.2f}")
    plt.title(f"corruption={row['strength']:.2f}")
    plt.legend(fontsize=8)

plt.tight_layout()
plt.savefig('ads4_phase_lock_v10_reconstruction_examples.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v10_reconstruction_examples.png')
